# 02 Linear Regression - House Price Prediction

This is a complete, standalone pipeline for training a Linear Regression model on dirty house price data. It handles unit normalization, missing values, and outlier capping.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import re
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

sns.set(style='whitegrid')

### 1. Data Cleaning & Preprocessing
Handling the "dirty" data discovered in EDA: Unit mismatches in `Lot_Size` and mixed-type `Bedrooms`.

In [ ]:
def clean_house_data(df):
    df = df.copy().dropna(subset=['Price']).drop_duplicates()
    
    # 1. Normalize Lot_Size (Acres to Sqft)
    def parse_lot(v):
        if pd.isna(v): return 10000 # Median fallback
        v = str(v).lower()
        num = float(re.findall(r'\d+\.\d+|\d+', v)[0])
        if 'ac' in v: return num * 43560
        return num
    df['Lot_Size'] = df['Lot_Size'].apply(parse_lot)
    
    # 2. Clean Bedrooms (handling '3+1' strings)
    df['Bedrooms'] = df['Bedrooms'].apply(lambda x: sum([int(i) for i in str(x).split('+')]) if '+' in str(x) else int(re.sub(r'\D', '', str(x))))
    
    # 3. Cap Outliers (Prices > 99th percentile)
    df['Price'] = df['Price'].clip(upper=df['Price'].quantile(0.99))
    
    return df

df = clean_house_data(pd.read_csv('../_data/house_prices.csv'))
X = df.drop('Price', axis=1)
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### 2. Pipeline Building & Training
We use a `ColumnTransformer` to handle numerical and categorical features separately.

In [ ]:
numeric_features = ['Living_Area', 'Lot_Size', 'Bedrooms', 'Bathrooms', 'Garage_Capacity', 'Year_Built']
categorical_features = ['Neighborhood', 'Property_Type', 'House_Style']

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='missing')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
])

model_pipeline = Pipeline([('preprocessor', preprocessor), ('regressor', LinearRegression())])
model_pipeline.fit(X_train, y_train)
print("Linear Regression model trained.")

### 3. Evaluation & Visualization

In [ ]:
y_pred = model_pipeline.predict(X_test)
print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")
print(f"MAE: ${mean_absolute_error(y_test, y_pred):.2f}")

plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.5, color='teal')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title('Actual vs Predicted House Prices')
plt.show()

### 4. Model Saving

In [ ]:
joblib.dump(model_pipeline, '../_model/house_prediction_linear_regression.joblib')
print("Model saved to src/ml/_model/")